In [ ]:
# scripts/migrate_figures.py
"""One-time cleanup: archive orphaned pre-manifest figures, since regenerating
through viz/ is cheap (cached pipeline) and safer than trying to backfill
manifest entries for files whose provenance we can't verify."""

import json
import shutil
from pathlib import Path

FIGURES_DIR = Path("./figures")
ARCHIVE_DIR = FIGURES_DIR / "_archive_pre_manifest"

def migrate():
    manifest_path = FIGURES_DIR / "manifest.json"
    known_paths = set()
    if manifest_path.exists():
        entries = json.loads(manifest_path.read_text())
        known_paths = {FIGURES_DIR / e["path"] for e in entries}

    ARCHIVE_DIR.mkdir(exist_ok=True)
    moved = 0
    for f in FIGURES_DIR.rglob("*.png"):
        if ARCHIVE_DIR in f.parents:
            continue
        if f not in known_paths:
            dest = ARCHIVE_DIR / f.relative_to(FIGURES_DIR)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(f), str(dest))
            moved += 1
    print(f"Archived {moved} unmanifested figure(s) to {ARCHIVE_DIR}")

In [ ]:
migrate()